In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 3, Finished, Available, Finished, False)

In [2]:
INITIAL_LOAD_DATE = "2026-01-01"   # the date the dimension was first loaded
SNAPSHOT_DATE = "2026-06-01"       # the date THIS update snapshot arrived
FAR_FUTURE_DATE = "9999-12-31" 

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 4, Finished, Available, Finished, False)

In [3]:
bronze_trades = (spark.read
                  .option("header", "true")
                  .option("inferSchema", "true")
                  .csv("Files/trade_blotter.csv"))

bronze_dim_trader = (spark.read
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .csv("Files/dim_trader.csv"))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 5, Finished, Available, Finished, False)

In [4]:
print(f"Bronze trades: {bronze_trades.count()} rows")
print(f"Bronze dim_trader: {bronze_dim_trader.count()} rows")
display(bronze_trades.limit(2))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 6, Finished, Available, Finished, False)

Bronze trades: 567 rows
Bronze dim_trader: 6 rows


SynapseWidget(Synapse.DataFrame, ec9b4e21-c0dd-46e5-a700-c8d85e2f5514)

In [5]:
## Keep only newest records
dedupe_window = Window.partitionBy("trade_id").orderBy(F.col("last_modified_ts").desc())

silver_trades = (bronze_trades
                  .withColumn("row_number", F.row_number().over(dedupe_window))
                  .filter(F.col("row_number") == 1)
                  .drop("row_number"))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 7, Finished, Available, Finished, False)

In [6]:
# drop corrput data
silver_trades = silver_trades.filter(F.col("quantity") > 0).filter(F.col("price") > 0)

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 8, Finished, Available, Finished, False)

In [7]:
silver_trades = silver_trades.withColumn(
    "trade_value", F.round(F.col("quantity") * F.col("price"), 2)
)

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 9, Finished, Available, Finished, False)

In [8]:
print(f"Bronze had {bronze_trades.count()} rows -> Silver has {silver_trades.count()} rows")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 10, Finished, Available, Finished, False)

Bronze had 567 rows -> Silver has 554 rows


In [9]:
silver_trades.write.mode("overwrite").format("delta").saveAsTable("silver_trades")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 11, Finished, Available, Finished, False)

In [10]:
## Gold layer

## Unique and aggregated data ready for consumption
## Aggregated on trader_id per day's values on each trade type

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 12, Finished, Available, Finished, False)

In [11]:
gold_fact_trades_daily = (silver_trades
                           .groupBy("trade_date", "trader_id", "trade_type")
                           .agg(
                               F.sum("trade_value").alias("total_trade_value"),
                               F.count("*").alias("trade_count"),
                           ))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 13, Finished, Available, Finished, False)

In [12]:
print(f"Gold fact_trades_daily: {gold_fact_trades_daily.count()} rows")

gold_fact_trades_daily.write.mode("overwrite").format("delta").saveAsTable("gold_fact_trades_daily")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 14, Finished, Available, Finished, False)

Gold fact_trades_daily: 290 rows


In [15]:
# Changing data

## Updated dimensions: SCD2

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 17, Finished, Available, Finished, False)

In [14]:
gold_dim_trader = (bronze_dim_trader
                        .withColumn("valid_from", F.lit(INITIAL_LOAD_DATE).cast("date"))
                        .withColumn("valid_to", F.lit(FAR_FUTURE_DATE).cast("date"))
                        .withColumn("is_current", F.lit(True)))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 16, Finished, Available, Finished, False)

In [16]:
display(gold_dim_trader.orderBy("trader_id"))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 541cdbaf-edce-4459-b51e-9c566eb456f0)

In [18]:
## Fresh data coming in for a specific date
# Filename dim_trader_snapshot_{SNAPSHOT_DATE}.csv
# Read data from here, and make these records as active 

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 20, Finished, Available, Finished, False)

In [19]:
incoming_snapshot = (spark.read
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .csv(f"Files/dim_trader_snapshot_{SNAPSHOT_DATE.replace('-', '')}.csv")
                      .drop("snapshot_date"))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 21, Finished, Available, Finished, False)

In [20]:
current_rows = gold_dim_trader.filter(F.col("is_current") == True)

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 22, Finished, Available, Finished, False)

In [21]:
comparison = (current_rows.alias("old")
              .join(incoming_snapshot.alias("new"), on="trader_id", how="outer")
              .filter(
                  F.col("old.trader_id").isNull() |             # brand new trader
                  (F.col("old.desk") != F.col("new.desk")) |     # desk changed
                  (F.col("old.region") != F.col("new.region"))   # region changed
              ))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 23, Finished, Available, Finished, False)

In [22]:
comparison.show(5)

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 26, Finished, Available, Finished, False)

+---------+-----------+--------+------+----------+----------+----------+-----------+--------+------+
|trader_id|trader_name|    desk|region|valid_from|  valid_to|is_current|trader_name|    desk|region|
+---------+-----------+--------+------+----------+----------+----------+-----------+--------+------+
|        4|     N. Rao|   Macro|  APAC|2026-01-01|9999-12-31|      true|     N. Rao|  Credit|  APAC|
|        5|  P. Sharma|  Credit|  APAC|2026-01-01|9999-12-31|      true|  P. Sharma|   Macro|  APAC|
|        6|    V. Nair|Equities|  APAC|2026-01-01|9999-12-31|      true|    V. Nair|Equities|  EMEA|
|        7|       NULL|    NULL|  NULL|      NULL|      NULL|      NULL|   K. Desai|Equities|  APAC|
+---------+-----------+--------+------+----------+----------+----------+-----------+--------+------+



In [23]:
changed_trader_ids = [row.trader_id for row in
                      comparison.select(F.coalesce("old.trader_id", "new.trader_id").alias("trader_id"))
                      .collect()]

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 28, Finished, Available, Finished, False)

In [24]:
print(f"Traders with a change today: {sorted(changed_trader_ids)}")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 29, Finished, Available, Finished, False)

Traders with a change today: [4, 5, 6, 7]


In [25]:
closed_out_rows = (current_rows
                    .filter(F.col("trader_id").isin(changed_trader_ids))
                    .withColumn("valid_to", F.date_sub(F.lit(SNAPSHOT_DATE).cast("date"), 1))
                    .withColumn("is_current", F.lit(False)))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 30, Finished, Available, Finished, False)

In [27]:
untouched_rows = current_rows.filter(~F.col("trader_id").isin(changed_trader_ids))
already_historical_rows = gold_dim_trader.filter(F.col("is_current") == False) 

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 32, Finished, Available, Finished, False)

In [28]:
new_current_rows = (incoming_snapshot
                     .filter(F.col("trader_id").isin(changed_trader_ids))
                     .withColumn("valid_from", F.lit(SNAPSHOT_DATE).cast("date"))
                     .withColumn("valid_to", F.lit(FAR_FUTURE_DATE).cast("date"))
                     .withColumn("is_current", F.lit(True)))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 33, Finished, Available, Finished, False)

In [29]:
gold_dim_trader_scd2 = (already_historical_rows
                         .unionByName(untouched_rows)
                         .unionByName(closed_out_rows)
                         .unionByName(new_current_rows))

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 34, Finished, Available, Finished, False)

In [30]:
gold_dim_trader_scd2.write.mode("overwrite").format("delta").saveAsTable("gold_dim_trader_scd2")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 35, Finished, Available, Finished, False)

In [31]:
print(f"\nRow count: {gold_dim_trader_scd2.count()} (started with {bronze_dim_trader.count()})")

StatementMeta(, c3aa6b93-f1d6-4b1b-aa22-912b053a4b6a, 36, Finished, Available, Finished, False)


Row count: 10 (started with 6)
